### 1. Import modules

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

### 2. Define path to data and load data

In [ ]:
data_path = 'C:\\Users\\daidd\\Documents\\GitHub\\601-Proj\\data\\data1.csv' # Make sure to define your own path to data
data = pd.read_csv(data_path)

### 3. Filter event (condition) and define outcomes

In [4]:
# Condition: age >= 18
data = data[data['r_age_at_event'] >= 18].copy()

# Outcome a)
days_10_years = 3652.5
data['stroke_10y'] = data['c_20yPost_avc_like_date_time_of_condition_assigned_days_from_reference'] <= days_10_years
data['hf_10y'] = data['c_20yPost_congestive_heart_failure_date_time_of_condition_assigned_days_from_reference'] <= days_10_years
data['ckd_10y'] = data['c_20yPost_chonic_kidney_disease_date_time_of_condition_assigned_days_from_reference'] <= days_10_years

# Combine into single bin outcome A (1 if any occured, else 0)
data['outcome_a_stroke_hf_ckd_10y'] = (data['stroke_10y'] | data['hf_10y'] | data['ckd_10y']).astype(int)

# Outcome b)
days_1_year = 365.25
data['outcome_b_ed_1y'] = (data['e_30dPost_admission_start_date_days_from_reference'] <= days_1_year).astype(int)

### 4. Select features 

In [5]:
features = [
    'birth_date',
    'gender',
    'l_90dPost_hba1c_result_numeric_calculated',
    'l_90dPost_cholesterol_total_result_numeric_calculated',
    'l_90dPost_glucose_result_numeric_calculated',
    'l_90dPost_homocystine_result_numeric_calculated',
    'l_90dPost_triglycerides_result_numeric_calculated',
    'l_90dPost_creatinine_result_numeric_calculated'
]

# Extract the features we need
X = data[features].copy()

### 5. Handle categorical data (gender) and missing numerical data

In [6]:
X['gender'] = X['gender'].map({'Male': 0, 'Female': 1})

# Fill any completely missing gender values with the most frequent value (mode)
if X['gender'].isnull().any():
    X['gender'] = X['gender'].fillna(X['gender'].mode()[0])

# Impute missing lab values with the median of each respective column
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

### 6. Rename col for clarity

In [ ]:
feature_names_map = {
    'birth_date': 'Year of Birth',
    'gender': 'Sex at Birth',
    'l_90dPost_hba1c_result_numeric_calculated': 'HbA1c',
    'l_90dPost_cholesterol_total_result_numeric_calculated': 'Cholesterol',
    'l_90dPost_glucose_result_numeric_calculated': 'Glycemia (Glucose)',
    'l_90dPost_homocystine_result_numeric_calculated': 'Homocysteine',
    'l_90dPost_triglycerides_result_numeric_calculated': 'Triglycerides',
    'l_90dPost_creatinine_result_numeric_calculated': 'Creatinine'
}

X_imputed.rename(columns=feature_names_map, inplace=True)

### 7. Combine features and outcomes into a final cleaned dataset and save to csv

In [8]:
y_a = data['outcome_a_stroke_hf_ckd_10y'].reset_index(drop=True)
y_b = data['outcome_b_ed_1y'].reset_index(drop=True)

cleaned_df = pd.concat([X_imputed, y_a, y_b], axis=1)

cleaned_df.to_csv('C:\\Users\\daidd\\Documents\\GitHub\\601-Proj\\data\\cleaned_diabetic_patients_data.csv', index=False)